# Kandinsky ImageSR 2.0 — inference example

Super-resolves one image with `KandinskyImageSRPipeline` and saves the result. The model bundle (transformer and VAE) is downloaded into the HF cache on first use.

Install the package first: `just setup` in this repository, or `pip install "kandinsky-image-sr @ git+https://github.com/kandinskylab/kandinsky-image-sr.git"` elsewhere. For FlashAttention-2 add the `flash` extra (`just setup flash`); without it the notebook runs on PyTorch SDPA.

### 1. Config

`ATTENTION = AttentionBackend.AUTO` uses FlashAttention-2 when `flash-attn` is installed and the device is CUDA, and PyTorch SDPA otherwise. `FLASH` fails instead of falling back; `NATIVE` always uses SDPA.

Set either `CREATIVITY` (calibrated level `0..8`, level 4 is the default) or `ONE_STEP_T`, not both; with both `None` the per-factor default is used.

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import torch
from IPython.display import display
from PIL import Image

from kandinsky_image_sr import KandinskyImageSRPipeline
from kandinsky_image_sr.attention import AttentionBackend, configure_attention

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_ID = "kandinskylab/Kandinsky-ImageSR-2.0"
INPUT_IMAGE = Path("inputs/lq.png")
OUT_DIR = Path("outputs")

DEVICE = torch.device("cuda")
UPSCALE_FACTOR = 4  # 2 or 4
CREATIVITY = None  # 0..8, or None for the default
ONE_STEP_T = None  # raw timestep, mutually exclusive with CREATIVITY
ATTENTION = AttentionBackend.AUTO

OUT_DIR.mkdir(parents=True, exist_ok=True)

### 2. Load the input image

In [ ]:
lq_image = Image.open(INPUT_IMAGE).convert("RGB")
print(f"input: {lq_image.width}x{lq_image.height}")
display(lq_image)

### 3. Load the pipeline

`configure_attention` switches the transformer to FlashAttention-2 or PyTorch SDPA and returns the backend actually in use.

In [ ]:
pipe = KandinskyImageSRPipeline.from_pretrained(MODEL_ID, dtype=torch.bfloat16).to(DEVICE)
backend = configure_attention(pipe.transformer, ATTENTION, DEVICE)
print(f"attention backend: {backend.value} (requested: {ATTENTION.value})")

### 4. Inference

The image is processed in overlapping 1024-pixel output tiles that are blended back with Hanning windows.

In [ ]:
start = time.perf_counter()
with torch.inference_mode():
    result = pipe(lq_image, upscale_factor=UPSCALE_FACTOR, creativity=CREATIVITY, one_step_t=ONE_STEP_T)
elapsed = time.perf_counter() - start

sr_image = result.images[0]
sr_path = OUT_DIR / f"{INPUT_IMAGE.stem}_sr_x{UPSCALE_FACTOR}.png"
sr_image.save(sr_path)
print(f"SR output: {sr_image.width}x{sr_image.height} in {elapsed:.2f} s -> {sr_path}")

### 5. Compare

A central crop of the bicubic upscale (left) next to the same crop of the SR output (right).

In [ ]:
CROP = min(512, sr_image.width, sr_image.height)  # crop side in output pixels

left = (sr_image.width - CROP) // 2
top = (sr_image.height - CROP) // 2
box = (left, top, left + CROP, top + CROP)
bicubic = lq_image.resize(sr_image.size, Image.Resampling.BICUBIC)

comparison = Image.new("RGB", (2 * CROP, CROP))
comparison.paste(bicubic.crop(box), (0, 0))
comparison.paste(sr_image.crop(box), (CROP, 0))
display(comparison)